In [4]:
import numpy as np
import torch 
import torch.nn as nn # neural network

In [13]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=8, layers=[32, 16, 8]):
        super().__init__()
        
        # Embedding tables P and Q (from Equation 3)
        self.user_embedding = nn.Embedding(n_users, embedding_dim)   # P
        self.item_embedding = nn.Embedding(n_items, embedding_dim)   # Q
        
        # Neural CF layers (the tower in Equation 4)
        self.layers = nn.ModuleList()
        input_size = embedding_dim * 2  # concatenate user + item
        
        for layer_size in layers:
            self.layers.append(nn.Linear(input_size, layer_size))
            input_size = layer_size
        
        # Output layer φ_out
        self.output_layer = nn.Linear(input_size, 1)
        
    def forward(self, user_ids, item_ids):
        # Get embeddings: P^T v_u and Q^T v_i
        user_emb = self.user_embedding(user_ids)      # shape: (batch, embedding_dim)
        item_emb = self.item_embedding(item_ids)
        
        # Concatenate (this is the input to φ1)
        x = torch.cat([user_emb, item_emb], dim=1)
        
        # Pass through all neural CF layers φ1 → φ2 → ... → φX
        for layer in self.layers:
            x = layer(x)
            x = torch.relu(x)          # paper uses ReLU
        
        # Final output layer φ_out + sigmoid
        out = self.output_layer(x)
        return torch.sigmoid(out)      # score between 0 and 1

In [17]:
model = NCF(n_users=10000, n_items=5000, embedding_dim=16, layers=[200, 100, 64, 32, 16])

# Fake batch: 4 users and 4 items
user_batch = torch.tensor([0, 1, 2, 3])
item_batch = torch.tensor([10, 20, 30, 40])

score = model(user_batch, item_batch)
print("Predicted scores:\n", score)

Predicted scores:
 tensor([[0.5420],
        [0.5433],
        [0.5415],
        [0.5428]], grad_fn=<SigmoidBackward0>)
